In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# Load validation data (has Age and Gender columns)
val_df = pd.read_parquet("../data/processed/val_final.parquet")

# Load the uncertainty/prediction results we saved (from LSTM model)
mean_preds = np.load("../reports/mean_predictions.npy")

print(val_df.shape)
print(len(mean_preds))

(230301, 73)
224710


In [2]:
# Recreate the same patient-level truncation that was applied when building sequences
# (max_len=100 hours per patient, same as during training)

def truncate_patient_data(data, max_len=100):
    truncated_rows = []
    for pid in data["patient_id"].unique():
        patient_data = data[data["patient_id"] == pid].sort_values("ICULOS")
        if len(patient_data) > max_len:
            patient_data = patient_data.iloc[:max_len]
        truncated_rows.append(patient_data)
    return pd.concat(truncated_rows, ignore_index=True)

# Apply this to validation data to match the predictions
val_df_aligned = truncate_patient_data(val_df, max_len=100)

print("Aligned val_df shape:", val_df_aligned.shape)
print("Predictions length:", len(mean_preds))

Aligned val_df shape: (224710, 73)
Predictions length: 224710


In [3]:
# Add predictions as a new column in the aligned dataframe
val_df_aligned = val_df_aligned.copy()
val_df_aligned["predicted_risk"] = mean_preds

# Create age groups for analysis
val_df_aligned["age_group"] = pd.cut(val_df_aligned["Age"], 
                                       bins=[0, 40, 60, 80, 120], 
                                       labels=["<40", "40-60", "60-80", "80+"])

# Check how many rows fall into each age group
print(val_df_aligned["age_group"].value_counts())

# Check gender distribution (0 and 1 in this dataset)
print("\nGender distribution:")
print(val_df_aligned["Gender"].value_counts())

age_group
60-80    99500
40-60    70747
80+      30256
<40      24207
Name: count, dtype: int64

Gender distribution:
Gender
1    124694
0    100016
Name: count, dtype: int64


In [4]:
# Calculate AUROC separately for each age group
print("=== Performance by Age Group ===")
for group in ["<40", "40-60", "60-80", "80+"]:
    subset = val_df_aligned[val_df_aligned["age_group"] == group]
    
    # Only calculate if there are both sepsis and non-sepsis cases in this group
    if subset["SepsisLabel"].nunique() > 1:
        auroc_group = roc_auc_score(subset["SepsisLabel"], subset["predicted_risk"])
        sepsis_pct = subset["SepsisLabel"].mean() * 100
        print(f"{group}: AUROC = {auroc_group:.4f}, n = {len(subset)}, sepsis% = {sepsis_pct:.2f}%")
    else:
        print(f"{group}: Not enough sepsis cases to calculate AUROC")

=== Performance by Age Group ===
<40: AUROC = 0.7987, n = 24207, sepsis% = 1.64%
40-60: AUROC = 0.7870, n = 70747, sepsis% = 1.37%
60-80: AUROC = 0.7614, n = 99500, sepsis% = 1.52%
80+: AUROC = 0.7843, n = 30256, sepsis% = 1.57%


In [5]:
print("=== Performance by Gender ===")
for gender in [0, 1]:
    subset = val_df_aligned[val_df_aligned["Gender"] == gender]
    
    if subset["SepsisLabel"].nunique() > 1:
        auroc_gender = roc_auc_score(subset["SepsisLabel"], subset["predicted_risk"])
        sepsis_pct = subset["SepsisLabel"].mean() * 100
        gender_label = "Female" if gender == 0 else "Male"
        print(f"{gender_label} (Gender={gender}): AUROC = {auroc_gender:.4f}, n = {len(subset)}, sepsis% = {sepsis_pct:.2f}%")

=== Performance by Gender ===
Female (Gender=0): AUROC = 0.7905, n = 100016, sepsis% = 1.51%
Male (Gender=1): AUROC = 0.7664, n = 124694, sepsis% = 1.49%


In [6]:
import json

fairness_results = {
    "age_group_auroc": {
        "<40": 0.7987,
        "40-60": 0.7870,
        "60-80": 0.7614,
        "80+": 0.7843
    },
    "gender_auroc": {
        "Female": 0.7905,
        "Male": 0.7664
    },
    "max_age_gap": 0.7987 - 0.7614,  # difference between best and worst age group
    "gender_gap": 0.7905 - 0.7664
}

with open("../reports/fairness_results.json", "w") as f:
    json.dump(fairness_results, f, indent=2)

print("Fairness results saved!")
print("Max age-group AUROC gap:", fairness_results["max_age_gap"])
print("Gender AUROC gap:", fairness_results["gender_gap"])

Fairness results saved!
Max age-group AUROC gap: 0.0373
Gender AUROC gap: 0.02410000000000001
